In [2]:
# === Strict MITRE ATT&CK mapper: tactics + techniques + sub-techniques ========
from typing import Dict, Any, Iterable, List, Tuple
import json, re, collections
import pandas as pd
from stix2 import MemoryStore, Filter, parse

# --------------------------- STIX loader --------------------------------------
def load_attack_store(path: str) -> MemoryStore:
    with open(path, "r") as f:
        raw = json.load(f)
    try:
        bundle = parse(raw, allow_custom=True)
        objs = bundle.objects
    except Exception:
        objs = raw.get("objects", raw)
    return MemoryStore(stix_data=objs)

def _get_attack_external_id(obj: Dict[str, Any], source_name: str = "mitre-attack"):
    for ref in obj.get("external_references", []) or []:
        src = ref.get("source_name")
        ext_id = ref.get("external_id")
        if src == source_name and ext_id:
            return ext_id
    return None

ID_TECH = re.compile(r"^T\d{4}(\.\d{3})?$", re.IGNORECASE)
ID_TAC  = re.compile(r"^TA\d{4}$", re.IGNORECASE)

# Strong normaliser for names/aliases (handles slash/dash spacing)
def norm_key(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"\s*/\s*", "/", s)   # "A / B" -> "A/B"
    s = re.sub(r"\s*-\s*", "-", s)   # "A - B" -> "A-B"
    s = re.sub(r"\s+", " ", s)       # collapse multiple spaces
    return s

# ---- Canonical alias dictionaries (extend as needed) -------------------------
# Technique aliases: norm_key(keys) → canonical technique NAME or TECH ID.
TECH_ALIAS_TO_CANON = {
    # Your unmatched set + common legacy phrasings
    "signed binary proxy execution": "System Binary Proxy Execution",   # T1218 (parent)
    "network service scanning": "Network Service Discovery",            # T1046
    "registry run keys": "T1547.001",                                   # map straight to subtech ID
    "credential dumping": "OS Credential Dumping",                      # T1003
    "remote file copy": "Ingress Tool Transfer",                        # T1105
    "scripting": "Command and Scripting Interpreter",                   # T1059
    "data exfiltration over web service": "Exfiltration Over Web Service",  # T1567

    # Useful short forms (map tools to umbrella names)
    "powershell": "Command and Scripting Interpreter",                  # T1059
    "windows command shell": "Command and Scripting Interpreter",       # T1059
    "bash": "Command and Scripting Interpreter",                        # T1059
    "wmic": "Windows Management Instrumentation",                       # T1047
    "regsvr32": "System Binary Proxy Execution",                        # T1218
    "mshta": "System Binary Proxy Execution",                           # T1218
}

# Tactic aliases: norm_key(keys) → canonical tactic NAME.
TACTIC_ALIAS_TO_CANON = {
    "defense evasion": "Defense Evasion",
    "defence evasion": "Defense Evasion",
    "privilege escalation": "Privilege Escalation",
    "credential access": "Credential Access",
    "command and control": "Command and Control",
    "initial access": "Initial Access",
    "lateral movement": "Lateral Movement",
    "collection": "Collection",
    "discovery": "Discovery",
    "exfiltration": "Exfiltration",
    "execution": "Execution",
    "impact": "Impact",
    "persistence": "Persistence",
    "reconnaissance": "Reconnaissance",
    "resource development": "Resource Development",
}

# --------------------------- Build indices ------------------------------------
def build_indices(stix_path: str):
    MS = load_attack_store(stix_path)
    techniques = MS.query([Filter("type", "=", "attack-pattern")])
    tactics    = MS.query([Filter("type", "=", "x-mitre-tactic")])
    rels       = MS.query([Filter("type", "=", "relationship")])

    # — Tactics —
    tac_name_to_id = {}
    tac_short_to_id = {}
    for t in tactics:
        name = t.get("name")
        short = t.get("x_mitre_shortname")
        taid = _get_attack_external_id(t)
        if name and taid:
            tac_name_to_id[name] = taid
        if short and taid:
            tac_short_to_id[short.lower()] = taid

    # — Techniques / sub-techniques —
    tech_name_to_id_active = {}
    tech_id_to_name_active = {}
    tech_id_is_sub = {}
    tech_id_to_parent = {}
    parent_to_subs = collections.defaultdict(list)

    # Legacy redirects (revoked/deprecated → replacement)
    legacy_name_to_new_id = {}
    legacy_id_to_new_id   = {}

    # STIX object id (uuid) → external technique id (Txxxx[.yyy])
    stixid_to_extid = {}
    for t in techniques:
        stixid_to_extid[t.get("id")] = _get_attack_external_id(t)

    # Build "revoked-by" external-ID redirects: old → new
    revoked_by_target = {}
    for r in rels:
        if r.get("relationship_type") == "revoked-by":
            old_ext = stixid_to_extid.get(r.get("source_ref"))
            new_ext = stixid_to_extid.get(r.get("target_ref"))
            if old_ext and new_ext:
                revoked_by_target[old_ext] = new_ext

    # Classify active vs legacy + build parent/sub maps
    for t in techniques:
        tid = _get_attack_external_id(t)
        if not tid:
            continue
        name = t.get("name")
        is_revoked = t.get("revoked", False)
        is_depr    = t.get("x_mitre_deprecated", False)
        is_sub     = t.get("x_mitre_is_subtechnique", False)
        parent = tid.split(".")[0] if is_sub and "." in tid else None

        if not is_revoked and not is_depr:
            tech_name_to_id_active[name] = tid
            tech_id_to_name_active[tid]  = name
            tech_id_is_sub[tid] = bool(is_sub)
            if parent:
                tech_id_to_parent[tid] = parent
                parent_to_subs[parent].append(tid)
        else:
            new_id = revoked_by_target.get(tid)
            if new_id:
                legacy_id_to_new_id[tid] = new_id
                if name:
                    legacy_name_to_new_id[name] = new_id

    # Parent-name helpers
    parent_name_to_parent_id = {name: tid for tid, name in tech_id_to_name_active.items() if "." not in tid}

    # Normalised maps (no fuzzy)
    tech_norm_to_id_active = {norm_key(k): v for k, v in tech_name_to_id_active.items()}
    parent_norm_to_parent_id = {norm_key(k): v for k, v in parent_name_to_parent_id.items()}

    return {
        "tac_name_to_id": tac_name_to_id,
        "tac_short_to_id": tac_short_to_id,
        "tech_name_to_id_active": tech_name_to_id_active,
        "tech_id_to_name_active": tech_id_to_name_active,
        "tech_id_is_sub": tech_id_is_sub,
        "tech_id_to_parent": tech_id_to_parent,
        "parent_to_subs": dict(parent_to_subs),
        "legacy_name_to_new_id": legacy_name_to_new_id,
        "legacy_id_to_new_id": legacy_id_to_new_id,
        "parent_name_to_parent_id": parent_name_to_parent_id,
        "tech_norm_to_id_active": tech_norm_to_id_active,
        "parent_norm_to_parent_id": parent_norm_to_parent_id,
    }

# --------------------------- Matching (no fuzzy) -------------------------------
def match_token(token: str, idx: Dict[str, Any],
                granularity: str = "parent",  # "parent" or "subtech"
                expand_parent_to_subs: bool = False):
    """
    Returns list of (attack_id, label_type, match_type).
      - label_type in {"tactic","technique"}
      - match_type in {"id","exact_name","shortname","alias","alias_id",
                       "alias_parent_expanded","legacy_redirect",
                       "parent","parent_expanded","unmatched"}
    """
    if not isinstance(token, str) or not token.strip():
        return [(None, None, "unmatched")]

    raw = token.strip()
    raw_norm = norm_key(raw)

    # 0) Already an ID?
    if ID_TAC.match(raw):
        return [(raw.upper(), "tactic", "id")]
    if ID_TECH.match(raw):
        tid = raw.upper()
        if tid in idx["legacy_id_to_new_id"]:
            return [(idx["legacy_id_to_new_id"][tid], "technique", "legacy_redirect")]
        return [(tid, "technique", "id")]

    # 1) Tactic (exact / shortname / alias)
    if raw in idx["tac_name_to_id"]:
        return [(idx["tac_name_to_id"][raw], "tactic", "exact_name")]
    short = raw_norm.replace(" ", "-")
    if short in idx["tac_short_to_id"]:
        return [(idx["tac_short_to_id"][short], "tactic", "shortname")]
    t_alias = TACTIC_ALIAS_TO_CANON.get(raw_norm)
    if t_alias and t_alias in idx["tac_name_to_id"]:
        return [(idx["tac_name_to_id"][t_alias], "tactic", "alias")]

    # 2) Technique exact (raw) or normalised name
    if raw in idx["tech_name_to_id_active"]:
        tid = idx["tech_name_to_id_active"][raw]
        if "." not in tid and granularity == "subtech" and expand_parent_to_subs:
            subs = idx["parent_to_subs"].get(tid, [])
            if subs:
                return [(s, "technique", "parent_expanded") for s in sorted(subs)]
        return [(tid, "technique", "exact_name")]

    tid_norm = idx["tech_norm_to_id_active"].get(raw_norm)
    if tid_norm:
        if "." not in tid_norm and granularity == "subtech" and expand_parent_to_subs:
            subs = idx["parent_to_subs"].get(tid_norm, [])
            if subs:
                return [(s, "technique", "parent_expanded") for s in sorted(subs)]
        return [(tid_norm, "technique", "exact_name")]

    # 2b) Technique alias → canonical NAME or direct ID
    canon_or_id = TECH_ALIAS_TO_CANON.get(raw_norm)
    if canon_or_id:
        # Alias maps directly to an ID (e.g., "T1547.001")
        if ID_TECH.match(canon_or_id):
            tid = canon_or_id.upper()
            # if the ID was deprecated, still let it through (or redirect if present)
            if tid in idx["legacy_id_to_new_id"]:
                tid = idx["legacy_id_to_new_id"][tid]
            return [(tid, "technique", "alias_id")]

        # Else alias maps to a canonical NAME → resolve (raw or normalised)
        if canon_or_id in idx["tech_name_to_id_active"]:
            tid = idx["tech_name_to_id_active"][canon_or_id]
        else:
            c_norm = norm_key(canon_or_id)
            tid = idx["tech_norm_to_id_active"].get(c_norm)

        if tid:
            if "." not in tid and granularity == "subtech" and expand_parent_to_subs:
                subs = idx["parent_to_subs"].get(tid, [])
                if subs:
                    return [(s, "technique", "alias_parent_expanded") for s in sorted(subs)]
            return [(tid, "technique", "alias")]

    # 3) Legacy (revoked/deprecated) NAME redirect
    if raw in idx["legacy_name_to_new_id"]:
        return [(idx["legacy_name_to_new_id"][raw], "technique", "legacy_redirect")]

    # 4) Parent name (exact or normalised)
    pid = idx["parent_name_to_parent_id"].get(raw) or idx["parent_norm_to_parent_id"].get(raw_norm)
    if pid:
        if granularity == "parent":
            return [(pid, "technique", "parent")]
        if expand_parent_to_subs:
            subs = idx["parent_to_subs"].get(pid, [])
            if subs:
                return [(s, "technique", "parent_expanded") for s in sorted(subs)]
        return [(pid, "technique", "parent")]

    # 5) Unmatched
    return [(None, None, "unmatched")]

# --------------------------- CSV pipeline -------------------------------------
def map_exercises_csv(
    exercises_csv: str,
    stix_path: str,
    ttp_col: str = "TTPs",       # OR set to None and provide separate cols below
    tactic_col: str = None,      # e.g., "Tactics"
    technique_col: str = None,   # e.g., "Techniques"
    subtech_col: str = None,     # e.g., "Subtechniques"
    exid_col: str = "EXID",
    sep_regex: str = r"[;,\|]+",
    granularity: str = "parent",         # "parent" or "subtech"
    expand_parent_to_subs: bool = False, # only used when granularity="subtech"
    out_map_csv: str = "ex_ttp_map_strict.csv",
    out_unmatched_csv: str = "ttps_unmatched.csv",
):
    idx = build_indices(stix_path)
    df = pd.read_csv(exercises_csv)
    if exid_col not in df.columns:
        raise ValueError(f"Missing `{exid_col}` in {exercises_csv}")

    splitter = re.compile(sep_regex)
    rows = []
    unmatched = collections.Counter()
    id_field = exid_col  # dynamic key name for output

    def yield_tokens(row) -> List[str]:
        tokens: List[str] = []
        if ttp_col and ttp_col in row and pd.notna(row[ttp_col]):
            tokens += [p.strip() for p in splitter.split(str(row[ttp_col])) if p.strip()]
        for c in (tactic_col, technique_col, subtech_col):
            if c and c in row and pd.notna(row[c]):
                tokens += [p.strip() for p in splitter.split(str(row[c])) if p.strip()]
        return tokens

    for _, r in df.iterrows():
        id_val = r[id_field]
        for tok in yield_tokens(r):
            matches = match_token(tok, idx, granularity=granularity,
                                  expand_parent_to_subs=expand_parent_to_subs)
            for attack_id, label_type, how in matches:
                if attack_id is None:
                    unmatched[tok] += 1
                rows.append({
                    id_field: id_val,             # <— dynamic column name here
                    "original_text": tok,
                    "label_type": label_type,     # "tactic" | "technique"
                    "attack_id": attack_id,       # TA0005 | T1003 | T1003.001
                    "match_type": how
                })

    out = pd.DataFrame(rows)
    out.to_csv(out_map_csv, index=False)

    um = pd.DataFrame([{"unmatched_text": k, "count": v} for k, v in unmatched.most_common()])
    um.to_csv(out_unmatched_csv, index=False)

    print(f"[OK] wrote {out_map_csv}  rows={len(out)}")
    print(f"[OK] wrote {out_unmatched_csv}  unique_unmatched={len(um)}")
    if not out.empty:
        print(out["match_type"].value_counts(dropna=False).to_string())


# --------------------------- Utility: subtech search ---------------------------
def find_subtechniques(parent_or_sub_id: str, stix_path: str, include_meta: bool = True):
    """
    Return all sub-techniques for a given ATT&CK technique ID.
    Accepts a parent ('T1547') or a sub-tech ('T1547.001') and normalises to the parent.
    """
    pid = parent_or_sub_id.strip().upper()
    if "." in pid:
        pid = pid.split(".", 1)[0]

    MS = load_attack_store(stix_path)
    techniques = MS.query([Filter("type", "=", "attack-pattern")])

    def _extid(t):
        for ref in t.get("external_references", []) or []:
            if ref.get("source_name") == "mitre-attack" and ref.get("external_id"):
                return ref["external_id"]
        return None

    rows = []
    for t in techniques:
        if not t.get("x_mitre_is_subtechnique"):
            continue
        tid = _extid(t)
        if not tid or not tid.startswith(pid + "."):
            continue
        name = t.get("name")
        tactics = [ph.get("phase_name") for ph in (t.get("kill_chain_phases") or [])]
        rows.append({"technique_id": tid, "name": name, "tactics": tactics})

    if not rows:
        print(f"[WARN] No sub-techniques found for {pid}")
        return pd.DataFrame(columns=["technique_id", "name", "tactics"]) if include_meta else []

    df = pd.DataFrame(rows).sort_values("technique_id").reset_index(drop=True)
    return df if include_meta else df["technique_id"].tolist()

def find_subtechniques_from_index(parent_id: str, idx: Dict[str, Any]) -> List[str]:
    """Fast lookup using existing parent_to_subs map from build_indices()."""
    parent = parent_id.split(".", 1)[0].upper()
    return sorted(idx["parent_to_subs"].get(parent, []))



In [3]:
EX_CSV = "exercises_full.csv"
STIX   = "attack-stix-data/enterprise-attack/enterprise-attack.json"

# 1) Map your exercises TTPs
map_exercises_csv(
    exercises_csv=EX_CSV,
    stix_path=STIX,
    ttp_col="ExTTPs",       # or tactic_col/technique_col/subtech_col
    exid_col="EXID",
    granularity="parent", # or "subtech"
    expand_parent_to_subs=True,
    out_map_csv="ex_ttp_map.csv",
    out_unmatched_csv="ex_ttps_unmatched.csv",
)

# 2) Quick subtech lookup
df = find_subtechniques("T1547", STIX, include_meta=True)
print(df.head())

[OK] wrote ex_ttp_map.csv  rows=135
[OK] wrote ex_ttps_unmatched.csv  unique_unmatched=0
match_type
exact_name    121
alias          12
alias_id        2
  technique_id                                name  \
0    T1547.001  Registry Run Keys / Startup Folder   
1    T1547.002              Authentication Package   
2    T1547.003                      Time Providers   
3    T1547.004                 Winlogon Helper DLL   
4    T1547.005           Security Support Provider   

                               tactics  
0  [persistence, privilege-escalation]  
1  [persistence, privilege-escalation]  
2  [persistence, privilege-escalation]  
3  [persistence, privilege-escalation]  
4  [persistence, privilege-escalation]  


In [4]:
ORG_CSV = "orgs_full.csv"
STIX   = "attack-stix-data/enterprise-attack/enterprise-attack.json"
map_exercises_csv(
    exercises_csv=ORG_CSV,
    stix_path=STIX,
    ttp_col="TTPs",       # or tactic_col/technique_col/subtech_col
    exid_col="ORGID",
    granularity="parent", # or "subtech"
    expand_parent_to_subs=True,
    out_map_csv="org_ttp_map.csv",
    out_unmatched_csv="org_ttps_unmatched.csv",
)

[OK] wrote org_ttp_map.csv  rows=245
[OK] wrote org_ttps_unmatched.csv  unique_unmatched=0
match_type
exact_name         220
alias               23
legacy_redirect      2
